<a href="https://colab.research.google.com/github/peremartra/Rearchitecting-LLMs/blob/main/CH07/CH07_NB04_uploadHF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<table style="width:100%; border:none; background:none;">
  <tr style="border:none;">
    <td style="border:none; vertical-align:middle; text-align:left; width: 120px;">
      <a href="https://hubs.la/Q040tvsK0"><img src="https://raw.githubusercontent.com/peremartra/Rearchitecting-LLMs/main/Images/cover.png" width="100px" style="border-radius: 4px;"></a>
    </td>
    <td style="border:none; vertical-align:middle; text-align:left;">
      <p style="margin: 0; font-size: 14px;">
        Supplementary code for the <a href="https://hubs.la/Q040tvsK0">Rearchitecting LLMs</a> book by <a href="https://github.com/peremartra">Pere Martra</a>.<br>
        <br>
        Code repository: <a href="https://github.com/peremartra/Rearchitecting-LLMs">https://github.com/peremartra/Rearchitecting-LLMs</a>
      </p>
    </td>
  </tr>
</table>

#Rearchitecting LLMs
## Structural techniques for efficient models


### Chapter 7: Specialization Tuning.

[![LinkedIn](https://img.shields.io/badge/LinkedIn-0077B5?style=flat&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/pere-martra/) [![GitHub](https://img.shields.io/badge/GitHub-100000?style=flat&logo=github&logoColor=white)](https://github.com/peremartra) [![X](https://img.shields.io/badge/X-000000?style=flat&logo=x&logoColor=white)](https://x.com/PereMartra) [![Hugging Face](https://img.shields.io/badge/🤗%20Hugging%20Face-blue)](https://huggingface.co/oopere)

_____
Colab Environment: GPU L4

Models:
* HuggingFaceTB/SmolLM2-1.7B-Instruct
_____

In this notebook we train a QLoRA adapter on the clinical NER dataset,
merge the adapter weights into the base model, generate a model card,
and publish the result to Hugging Face Hub under the authenticated user's profile.
Finally, we delete the local model from memory, reload it from the Hub,
and run an inference smoke-test to confirm the upload was successful.

# Setting up the notebook

In [1]:
# We pin versions to ensure reproducibility across different Colab sessions.
# peft provides LoRA adapters,
# trl gives us SFTTrainer, and huggingface_hub handles the upload workflow.
!pip install -q \
    transformers==5.0.0 \
    datasets==4.0.0 \
    peft==0.18.0 \
    trl==0.28.0 \
    bitsandbytes==0.49.2 \
    accelerate==1.12.0 \
    huggingface_hub \
    lm-eval \
    langdetect \
    optipfair \
    codecarbon

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 6.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 65.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 144.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 556.4/556.4 kB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 540.5/540.5 kB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 125.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 384.6/384.6 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import json
import os
import torch
import numpy as np

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from huggingface_hub import HfApi, login, whoami

In [3]:
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)

set_seed(42)

In [4]:
# The helper functions used in previous chapters are grouped in utils.py.
# We download it so the notebook is self-contained on Colab.
!wget -q https://raw.githubusercontent.com/peremartra/Rearchitecting-LLMs/main/utils.py

if os.path.exists('utils.py'):
    print('\u2705 utils.py downloaded successfully')
else:
    print('\u274c Failed to download utils.py')

from utils import clear_gpu_cache

✅ utils.py downloaded successfully


In [5]:
# ── Model & dataset ────────────────────────────────────────────────────────
MODEL_NAME      = 'HuggingFaceTB/SmolLM2-1.7B-Instruct'
DATASET_NAME    = 'oopere/clinical-ner-qdora'

# ── QLoRA hyperparameters ──────────────────────────────────────────────────
LORA_RANK       = 8
LORA_ALPHA      = 16
LORA_DROPOUT    = 0.05
TRAIN_EPOCHS    = 3
LEARNING_RATE   = 2e-4
BATCH_SIZE      = 8
MAX_SEQ_LENGTH  = 512

# ── HuggingFace Hub ────────────────────────────────────────────────────────
# Only the model name is set here — the username is resolved dynamically
# at upload time from the authenticated user's HF_TOKEN.
# This makes the notebook portable: any user who configures HF_TOKEN
# will upload the model to their own profile.
HF_MODEL_NAME   = 'SmolLM2-1.7B-ClinicalNER'
OUTPUT_DIR      = './qlora_merged'

# ── Runtime ───────────────────────────────────────────────────────────────
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU memory available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Device: cuda
GPU memory available: 23.7 GB


# 7.3.1 Base model and clinical dataset

In [6]:
dataset = load_dataset(DATASET_NAME)

print(f'Train samples : {len(dataset["train"])}')
print(f'Test samples  : {len(dataset["test"])}')
print()

train_df = dataset['train'].to_pandas()
print('Category distribution (train):')
print(train_df['category'].value_counts().to_string())

README.md:   0%|          | 0.00/4.90k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/102k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/17.9k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/360 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/40 [00:00<?, ? examples/s]

Train samples : 360
Test samples  : 40

Category distribution (train):
category
clean            92
abbreviations    74
implicit         73
typos            63
irrelevant       58


# QLoRA Training

We load the base model in 4-bit NF4 quantization, attach a LoRA adapter,
and fine-tune for 3 epochs on the clinical NER dataset.
The training mirrors exactly the setup used in **CH07_NB02_L4_QLoRA_QDoRA**.

In [7]:
print(f'Loading {MODEL_NAME}...')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map='auto',
)

print('Model loaded.')
print(f'GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

Loading HuggingFaceTB/SmolLM2-1.7B-Instruct...


config.json:   0%|          | 0.00/908 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Model loaded.
GPU memory used: 3.42 GB


### Schema validation

We define the expected JSON schema and a compliance checker.
This is reused in the final smoke-test after uploading to HF.

In [8]:
REQUIRED_SCHEMA = {
    'patient_age':   (int, type(None)),
    'symptoms':      list,
    'vital_signs':   dict,
    'medications':   list,
    'duration_days': (int, type(None)),
}

VITAL_SIGNS_KEYS = {'temperature', 'heart_rate', 'blood_pressure'}

def check_schema_compliance(response_text: str) -> dict:
    '''Evaluates whether a model response is a valid, schema-compliant JSON.'''
    cleaned = response_text.strip().removeprefix('```json').removesuffix('```').strip()
    try:
        parsed = json.loads(cleaned)
    except json.JSONDecodeError as e:
        return {'is_valid_json': False, 'matches_schema': False, 'failure_reason': f'JSON parse error: {e}'}

    for key, expected_type in REQUIRED_SCHEMA.items():
        if key not in parsed:
            return {'is_valid_json': True, 'matches_schema': False, 'failure_reason': f'Missing key: {repr(key)}'}
        if not isinstance(parsed[key], expected_type):
            return {'is_valid_json': True, 'matches_schema': False,
                    'failure_reason': f'Wrong type for {repr(key)}: got {type(parsed[key]).__name__}'}

    for vk in VITAL_SIGNS_KEYS:
        if vk not in parsed['vital_signs']:
            return {'is_valid_json': True, 'matches_schema': False, 'failure_reason': f'Missing vital_signs key: {repr(vk)}'}

    return {'is_valid_json': True, 'matches_schema': True, 'failure_reason': None}

In [9]:
# The minimal prompt is what the model learns to respond to after fine-tuning.
MINIMAL_PROMPT = 'Extract:'

In [10]:
def run_inference(note: str, system_prompt: str) -> str:
    '''Runs inference on a single clinical note using the current global model.'''
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user',   'content': note},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer([text], return_tensors='pt').to(model.device)
    with torch.inference_mode():
        output_ids = model.generate(**inputs, max_new_tokens=256, do_sample=False)
    new_tokens = output_ids[0][len(inputs.input_ids[0]):]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

In [11]:
# Training examples use the minimal prompt on the system side and the
# ground-truth JSON on the assistant side. This teaches the model to
# associate 'Extract:' with a schema-compliant JSON response.
def format_training_example(sample: dict) -> str:
    label = json.loads(sample['label']) if isinstance(sample['label'], str) \
            else sample['label']
    messages = [
        {'role': 'system',    'content': MINIMAL_PROMPT},
        {'role': 'user',      'content': sample['note']},
        {'role': 'assistant', 'content': json.dumps(label, indent=2)},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )

train_dataset = dataset['train'].map(
    lambda x: {'text': format_training_example(x)},
    remove_columns=dataset['train'].column_names,
)

print('Sample training example:')
print('-' * 60)
print(train_dataset[0]['text'][:500], '...')

Map:   0%|          | 0/360 [00:00<?, ? examples/s]

Sample training example:
------------------------------------------------------------
<|im_start|>system
Extract:<|im_end|>
<|im_start|>user
pt is 45 y.o male came in complainig of rly bad cogh and fevr for last 3 dys. says he feels super weak. took sum tyelnol 500mg evry 6 hrs but still feels sick. vitals r t 101.2 hr 98 bp 130/85 no othre medcations<|im_end|>
<|im_start|>assistant
{
  "patient_age": 45,
  "symptoms": [
    "cough",
    "fever",
    "weakness"
  ],
  "vital_signs": {
    "temperature": 101.2,
    "heart_rate": 98,
    "blood_pressure": "130/85"
  },
  "medicatio ...


## LoRA configuration

In [12]:
# prepare_model_for_kbit_training ensures the 4-bit quantized model
# is correctly set up for gradient computation through the adapter layers.
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    task_type='CAUSAL_LM',
    use_dora=False,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 9,043,968 || all params: 1,720,420,352 || trainable%: 0.5257


## Training QLoRA

In [13]:
training_args = SFTConfig(
    output_dir='./qlora_output',
    num_train_epochs=TRAIN_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=1,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type='cosine',
    warmup_steps=7,
    fp16=False,
    bf16=True,
    logging_steps=10,
    save_strategy='no',
    max_length=MAX_SEQ_LENGTH,
    dataset_text_field='text',
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    args=training_args,
)

print('Starting QLoRA training...')
trainer.train()
print('Training complete.')

Adding EOS to train dataset:   0%|          | 0/360 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/360 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/360 [00:00<?, ? examples/s]

Starting QLoRA training...


Step,Training Loss
10,1.385683
20,1.098856
30,0.893543
40,0.733532
50,0.648501
60,0.593425
70,0.576753
80,0.543361
90,0.540299
100,0.461376


Training complete.


# Merging adapter weights

`merge_and_unload()` folds the LoRA adapter deltas back into the base weight
matrices. The result is a standard `AutoModelForCausalLM` with no external
adapter files — ready to be pushed to the Hub and loaded by anyone with a
single `from_pretrained` call.

In [14]:
print('Merging QLoRA adapter weights into the base model...')
merged_model = model.merge_and_unload()

os.makedirs(OUTPUT_DIR, exist_ok=True)
merged_model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f'Merged model saved to {OUTPUT_DIR}')
print(f'GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

Merging QLoRA adapter weights into the base model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged model saved to ./qlora_merged
GPU memory used: 6.97 GB


# Generating the model card

We build a comprehensive `README.md` for the Hub repository.
Schema compliance results are hardcoded from the **CH07_NB02_L4_QLoRA_QDoRA**
experiments — no benchmarks are re-run here.

In [15]:
def generate_model_card(hf_repo_id, base_model, dataset_name):
    '''Builds the README.md string for the HF Hub repository.
    Schema compliance figures are taken from CH07_NB02_L4_QLoRA_QDoRA (SmolLM2-1.7B row).
    '''
    L = []

    # ── YAML frontmatter ──────────────────────────────────────────────────
    L += ['---', 'language:', '- en', 'tags:', '- qlora', '- clinical-ner',
          '- smollm2', '- rearchitecting-llms', '- fine-tuning', '- medical',
          '- educational', 'license: apache-2.0',
          'base_model: ' + base_model,
          'datasets:', '- ' + dataset_name, 'metrics:', '- accuracy', '---', '']

    # ── Title ─────────────────────────────────────────────────────────────
    L += ['# SmolLM2-1.7B-ClinicalNER', '',
          '## Model Description', '',
          'QLoRA fine-tuned version of **' + base_model + '** for clinical named entity',
          'recognition (NER). Created as part of **Chapter 7** of *Rearchitecting LLMs*.',
          '',
          '* **Book:** [Rearchitecting LLMs](https://hubs.la/Q04k2VyY0)',
          '* **Technique:** QLoRA (Quantized Low-Rank Adaptation)',
          '* **Task:** Clinical NER — structured JSON extraction from clinical notes',
          '* **Chapter:** Chapter 7 — Specialization Tuning',
          '',
          '[![Rearchitecting LLMs](https://cdn-uploads.huggingface.co/production/uploads/640f7924f2d7c41a1e9eced1/sa4ivCbm8kk6C9NAPmb-x.jpeg)](https://hubs.la/Q040tvsK0)',
          '', '---', '']

    # ── What it does ──────────────────────────────────────────────────────
    L += ['## What This Model Does', '',
          'Given a free-text clinical note, the model extracts structured clinical entities',
          'into a strict JSON schema using only the two-word prompt `Extract:`.',
          '',
          'Before fine-tuning: a 15-line system prompt was required.',
          'After QLoRA training: the model responds correctly to `Extract:` alone.',
          '', '---', '']

    # ── Schema compliance (hardcoded from NB02 SmolLM2-1.7B results) ──────
    L += ['## Schema Compliance Results', '',
          'Results from **CH07_NB02_L4_QLoRA_QDoRA** on the `' + dataset_name + '` test set (40 samples, 5 categories).',
          '',
          '| Model | Prompt | Schema Compliance |',
          '|:---|:---|:---:|',
          '| SmolLM2-1.7B baseline | Strict (15-line prompt) | 87.5% |',
          '| SmolLM2-1.7B baseline | Minimal (`Extract:`) | 0.0% |',
          '| **SmolLM2-1.7B QLoRA (this model)** | **Minimal (`Extract:`)** | **95.0%** |',
          '',
          'Fine-tuning permanently absorbed the 15-line prompt into the model weights.',
          '', '---', '']

    # ── Training details ──────────────────────────────────────────────────
    L += ['## Training Details', '', '### Dataset', '',
          '* **Source:** [' + dataset_name + '](https://huggingface.co/datasets/' + dataset_name + ')',
          '* **Train samples:** 200 (40 per category)',
          '* **Test samples:** 40 (8 per category)',
          '* **Categories:** clean, abbreviations, implicit, typos, irrelevant',
          '',
          '### QLoRA Hyperparameters', '',
          '| Parameter | Value |',
          '|:---|:---|',
          '| Base model | `' + base_model + '` |',
          '| Quantization | NF4 4-bit + double quantization |',
          '| LoRA rank (r) | 8 |',
          '| LoRA alpha | 16 |',
          '| LoRA dropout | 0.05 |',
          '| Target modules | q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj |',
          '| Epochs | 3 |',
          '| Batch size | 8 |',
          '| Learning rate | 2e-4 |',
          '| LR scheduler | cosine |',
          '| Max sequence length | 512 |',
          '| Compute dtype | bfloat16 |',
          '',
          '### Hardware',
          '* **GPU:** NVIDIA L4 (Google Colab)',
          '', '---', '']

    # ── How to use ────────────────────────────────────────────────────────
    L += ['## How to Use', '',
          '```python',
          'from transformers import AutoModelForCausalLM, AutoTokenizer',
          'import torch',
          '',
          'model_id = ' + repr(hf_repo_id),
          'tokenizer = AutoTokenizer.from_pretrained(model_id)',
          'model = AutoModelForCausalLM.from_pretrained(model_id)',
          'model.eval()',
          '',
          "note = 'Patient 45yo male, fever and dry cough for 3 days. Temp 38.5C, HR 98, BP 120/80.'",
          '',
          "messages = [{'role': 'system', 'content': 'Extract:'}, {'role': 'user', 'content': note}]",
          'text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)',
          "inputs = tokenizer([text], return_tensors='pt').to(model.device)",
          '',
          'with torch.inference_mode():',
          '    output_ids = model.generate(**inputs, max_new_tokens=256, do_sample=False)',
          '',
          'new_tokens = output_ids[0][len(inputs.input_ids[0]):]',
          'print(tokenizer.decode(new_tokens, skip_special_tokens=True))',
          '```',
          '', '---', '']

    # ── Limitations ───────────────────────────────────────────────────────
    L += ['## Limitations & Intended Use', '',
          '**Educational model** from *Rearchitecting LLMs* Chapter 7.',
          'Demonstrates QLoRA fine-tuning and the workflow: Train \u2192 Merge \u2192 Upload \u2192 Verify.',
          '',
          '**Not intended for clinical or production use.** Training data is synthetic.',
          '', '---', '']

    # ── Citation ──────────────────────────────────────────────────────────
    L += ['## Citation', '',
          '```bibtex',
          '@book{martra2026rearchitecting,',
          '  author    = {Pere Martra},',
          '  title     = {Rearchitecting LLMs: Structural techniques for efficient models},',
          '  publisher = {Manning Publications},',
          '  year      = {2026},',
          '  url       = {https://hubs.la/Q040tvtp0}',
          '}',
          '```',
          '', '---', '']

    # ── Acknowledgments ───────────────────────────────────────────────────
    L += ['## Acknowledgments', '',
          'Created following *Rearchitecting LLMs* (Manning, 2026).',
          'Challenge: can you push schema compliance above 95%?',
          'Try: higher LoRA rank, more epochs, or QDoRA instead.',
          'Share your results: [discussion forum](https://hubs.la/Q04k2VyY0)',
          '____']

    return '\n'.join(L)

# Uploading to Hugging Face

Authentication reads `HF_TOKEN` from Colab secrets first, falls back to the
`HF_TOKEN` environment variable, and finally uses cached credentials.
The upload repository is resolved dynamically from the authenticated user's
profile — **no username is hardcoded**.

In [16]:
# ── Authentication ────────────────────────────────────────────────────────
# Colab secrets (recommended) → environment variable → cached credentials
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print('\u2705 Authenticated via Colab secrets')
except Exception:
    hf_token = os.environ.get('HF_TOKEN')
    if hf_token:
        login(token=hf_token)
        print('\u2705 Authenticated via environment variable')
    else:
        login()  # Uses cached credentials or interactive prompt
        print('\u2705 Authenticated via cached credentials')

# ── Resolve username dynamically ──────────────────────────────────────────
HF_USERNAME = whoami()['name']
HF_REPO_ID  = f'{HF_USERNAME}/{HF_MODEL_NAME}'
print(f'\nWill upload to: https://huggingface.co/{HF_REPO_ID}')

✅ Authenticated via Colab secrets

Will upload to: https://huggingface.co/oopere/SmolLM2-1.7B-ClinicalNER


In [17]:
# ── Generate and write model card ─────────────────────────────────────────
card_content = generate_model_card(HF_REPO_ID, MODEL_NAME, DATASET_NAME)
readme_path  = os.path.join(OUTPUT_DIR, 'README.md')

with open(readme_path, 'w', encoding='utf-8') as f:
    f.write(card_content)

print(f'Model card saved to {readme_path}')
print()
print('Card preview (first 600 chars):')
print('-' * 50)
print(card_content[:600])

Model card saved to ./qlora_merged/README.md

Card preview (first 600 chars):
--------------------------------------------------
---
language:
- en
tags:
- qlora
- clinical-ner
- smollm2
- rearchitecting-llms
- fine-tuning
- medical
- educational
license: apache-2.0
base_model: HuggingFaceTB/SmolLM2-1.7B-Instruct
datasets:
- oopere/clinical-ner-qdora
metrics:
- accuracy
---

# SmolLM2-1.7B-ClinicalNER

## Model Description

QLoRA fine-tuned version of **HuggingFaceTB/SmolLM2-1.7B-Instruct** for clinical named entity
recognition (NER). Created as part of **Chapter 7** of *Rearchitecting LLMs*.

* **Book:** [Rearchitecting LLMs](https://hubs.la/Q04k2VyY0)
* **Technique:** QLoRA (Quantized Low-Rank Adaptation)
* **Task:** 


In [18]:
# ── Create repo and upload ────────────────────────────────────────────────
api = HfApi()

api.create_repo(
    repo_id=HF_REPO_ID,
    repo_type='model',
    exist_ok=True,       # Safe to re-run: won't fail if repo already exists
)
print(f'Repository ready: {HF_REPO_ID}')

api.upload_folder(
    folder_path=OUTPUT_DIR,
    repo_id=HF_REPO_ID,
    repo_type='model',
    commit_message='Upload SmolLM2-1.7B QLoRA fine-tuned on Clinical NER (Chapter 7)',
)

print(f'\n\u2705 Model uploaded successfully!')
print(f'   https://huggingface.co/{HF_REPO_ID}')

Repository ready: oopere/SmolLM2-1.7B-ClinicalNER


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ..._merged/model.safetensors:   0%|          | 15.9MB / 6.85GB            


✅ Model uploaded successfully!
   https://huggingface.co/oopere/SmolLM2-1.7B-ClinicalNER


# Verifying the upload

We delete both the PEFT model and the merged model from GPU memory, then
reload the model directly from the Hub. A single test sample from the dataset
is run through inference to confirm the upload was successful and the model
produces a schema-compliant JSON.

In [19]:
# merge_and_unload() returns the base model object (adapters inlined).
# Both 'model' and 'merged_model' reference the same object after the merge,
# so deleting one and calling gc + empty_cache is sufficient.
import gc

del merged_model
del model
gc.collect()
torch.cuda.empty_cache()

print('Local model deleted from memory.')
print(f'GPU memory after cleanup: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

Local model deleted from memory.
GPU memory after cleanup: 6.97 GB


In [20]:
# Reload in 4-bit to fit comfortably on the L4 GPU.
print(f'Loading {HF_REPO_ID} from Hugging Face Hub...')

hf_tokenizer = AutoTokenizer.from_pretrained(HF_REPO_ID)
hf_model = AutoModelForCausalLM.from_pretrained(
    HF_REPO_ID,
    #quantization_config=bnb_config,
    device_map='auto',
)
hf_model.eval()

print('Model reloaded from HF Hub.')
print(f'GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

Loading oopere/SmolLM2-1.7B-ClinicalNER from Hugging Face Hub...


config.json:   0%|          | 0.00/952 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/405 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.52M [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/368 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/6.85G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

Model reloaded from HF Hub.
GPU memory used: 13.82 GB


In [21]:
# Run a single inference call on a test sample to verify the round-trip.
# A schema-compliant JSON response confirms the model was correctly uploaded
# and the fine-tuning is preserved after the merge-and-upload workflow.

test_sample = dataset['test'][0]

print(f'Category : {test_sample["category"]}')
print(f'Note     : {test_sample["note"][:100]}')
print()

messages = [
    {'role': 'system', 'content': MINIMAL_PROMPT},
    {'role': 'user',   'content': test_sample['note']},
]
text   = hf_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = hf_tokenizer([text], return_tensors='pt').to(hf_model.device)

with torch.inference_mode():
    output_ids = hf_model.generate(**inputs, max_new_tokens=256, do_sample=False)

new_tokens = output_ids[0][len(inputs.input_ids[0]):]
response   = hf_tokenizer.decode(new_tokens, skip_special_tokens=True)

print('Model response:')
print(response)

compliance = check_schema_compliance(response)
print()
print(f'Schema compliance: {compliance}')

if compliance['matches_schema']:
    print('\u2705 Schema validation passed — model is working correctly from HF Hub!')
else:
    print(f'\u274c Schema validation failed: {compliance["failure_reason"]}')

Category : typos
Note     : 42 y.o male pt comin in c/o fevr and chst pain for like 3 days now. also has a bad cough. vitls r te

Model response:
{
  "patient_age": 42,
  "symptoms": [
    "fever",
    "chest pain",
    "bad cough"
  ],
  "vital_signs": {
    "temperature": 38.8,
    "heart_rate": 102,
    "blood_pressure": "130/85"
  },
  "medications": [
    {
      "name": "amoxicillin",
      "dose": "500mg",
      "frequency": "twice a day"
    },
    {
      "name": "tylenol",
      "dose": "not specified",
      "frequency": "not specified"
    }
  ],
  "duration_days": 3
}

Schema compliance: {'is_valid_json': True, 'matches_schema': True, 'failure_reason': None}
✅ Schema validation passed — model is working correctly from HF Hub!
